# Human-AI Interaction Reliance Quality Lab


## Purpose

This lab introduces reliance quality as a human-AI interaction metric.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)
n = 1200

df = pd.DataFrame({
    "case_id": [f"HIAI-{i:04d}" for i in range(1, n + 1)],
    "model_confidence": rng.beta(5, 2, size=n),
    "explanation_quality": rng.beta(4, 3, size=n),
    "uncertainty_clarity": rng.beta(3.5, 3, size=n),
    "user_expertise": rng.choice(["novice", "intermediate", "expert"], size=n, p=[0.30, 0.45, 0.25]),
    "risk_level": rng.choice(["low", "medium", "high"], size=n, p=[0.45, 0.35, 0.20]),
    "time_pressure": rng.choice(["low", "medium", "high"], size=n, p=[0.35, 0.40, 0.25]),
})

df["model_correct"] = rng.binomial(1, np.clip(0.20 + 0.75 * df["model_confidence"], 0, 1), size=n)
df.head()

In [ ]:
acceptance_probability = np.clip(
    0.10 + 0.55 * df["model_confidence"] + 0.15 * df["explanation_quality"],
    0.02,
    0.98,
)

df["user_accepted_ai"] = rng.binomial(1, acceptance_probability, size=n)
df["overreliance"] = ((df["user_accepted_ai"] == 1) & (df["model_correct"] == 0)).astype(int)
df["underreliance"] = ((df["user_accepted_ai"] == 0) & (df["model_correct"] == 1)).astype(int)
df["reliance_gap"] = abs(df["user_accepted_ai"] - df["model_correct"])

df.groupby(["user_expertise", "risk_level"]).agg(
    cases=("case_id", "count"),
    accuracy=("model_correct", "mean"),
    acceptance_rate=("user_accepted_ai", "mean"),
    overreliance_rate=("overreliance", "mean"),
    underreliance_rate=("underreliance", "mean"),
    reliance_gap=("reliance_gap", "mean"),
).reset_index()